In [1]:
"""
    기흥IC ~ 양재IC VDS 데이터 추출
"""

import json
import os
import pandas as pd

point_path = r"C:\Workspace\vissimTest_Python\network_version2\2025\resource\road_point.json"
name_path = r"C:\Workspace\vissimTest_Python\network_version2\2025\resource\route_name.json"

save_path = r"C:\Users\(주)내일이엔시 도로교통안전연구소\Desktop\프로젝트\2026년\1. 초장대 K-지하고속도로 인프라 안전 및 효율 향상 기술 개발\2. 원시데이터\VDS 데이터\기흥IC~양재IC(VDS).csv"

# 노선명 json 파일 읽어옴
with open(point_path, "r", encoding="utf-8") as point:
    road_point = json.load(point)

# 노선 json 파일 읽어옴
with open(name_path, "r", encoding="utf-8") as route:
    route_name = json.load(route)

folder_path = r"E:\2026년\1. 초장대 K-지하고속도로 인프라 안전 및 효율 향상 기술 개발\VISSIM_Workspace\vds"

csv_list = sorted([file for file in os.listdir(folder_path) if file.endswith(".csv")],  key=lambda x: int(x.split("_")[0]) if x.split("_")[0].isdigit() else 0)

result_list = []

for file_name in csv_list:
    full_path = os.path.join(folder_path, file_name)
    print("작업파일 : ", full_path)

    use_cols = ["수집일자", "수집시분초", "VDS_ID", "점유율", "평균속도", "교통량", "차로번호", "콘존명"]
    dtype_dict = {
        "교통량": "int32",
        "차로번호": "int8",
        "점유율": "float32",
        "평균속도": "float32"
    }
    df = pd.read_csv(full_path,  encoding="cp949", usecols=use_cols, dtype=dtype_dict)


    df["VDS"] = df["VDS_ID"].str.extract(r'(\d{4})')

    df["노선명"] = df["VDS"].map(route_name)


    #targets = ["서초IC","양재IC","금토JC","대왕판교IC","판교JC","판교IC","서울TG","신갈JC","수원신갈IC","기흥IC","동탄JC"]
    #filtered_df = df[(df["콘존명"].isin(targets)) & (df["노선명"] == "경부선")]

    filtered_df =  df[df["콘존명"].str.contains("서초IC|양재IC|금토JC|대왕판교IC|판교JC|판교IC|서울TG|신갈JC|수원신갈IC|기흥IC|동탄JC", na=False) & (df["노선명"] == "경부선")]

    # 예시: 속도값이 -1~10인 행 제거
    if "평균속도" in filtered_df.columns:
        filtered_df = filtered_df[~filtered_df["평균속도"].between(-1, 10)]

    filtered_df["시간대_num"] = (filtered_df["수집시분초"] // 10000) + 1
    filtered_df["시간대"] = filtered_df["시간대_num"].astype(str) + "시"

    agg_df = (
        filtered_df.groupby(
            ["수집일자", "시간대_num", "시간대", "VDS_ID", "콘존명", "차로번호"],
            as_index=False
        )
        .agg({
            "점유율": "mean",      # 평균
            "평균속도": "mean",    # 평균
            "교통량": "sum"        # 합계
        })
    )
    agg_df = agg_df.drop(columns=["시간대_num"], errors="ignore")
    result_list.append(agg_df)

final_df = pd.concat(result_list, ignore_index=True)

final_df.to_csv(save_path, index=False, encoding="utf-8-sig")
print("저장 완료")

작업파일 :  E:\2026년\1. 초장대 K-지하고속도로 인프라 안전 및 효율 향상 기술 개발\VISSIM_Workspace\vds\VDS_16_04_01_755333.csv
작업파일 :  E:\2026년\1. 초장대 K-지하고속도로 인프라 안전 및 효율 향상 기술 개발\VISSIM_Workspace\vds\VDS_16_04_01_855250.csv
작업파일 :  E:\2026년\1. 초장대 K-지하고속도로 인프라 안전 및 효율 향상 기술 개발\VISSIM_Workspace\vds\VDS_16_04_01_494979.csv
작업파일 :  E:\2026년\1. 초장대 K-지하고속도로 인프라 안전 및 효율 향상 기술 개발\VISSIM_Workspace\vds\VDS_16_04_01_432020.csv
작업파일 :  E:\2026년\1. 초장대 K-지하고속도로 인프라 안전 및 효율 향상 기술 개발\VISSIM_Workspace\vds\VDS_16_04_01_861140.csv
작업파일 :  E:\2026년\1. 초장대 K-지하고속도로 인프라 안전 및 효율 향상 기술 개발\VISSIM_Workspace\vds\VDS_16_04_01_591143.csv
작업파일 :  E:\2026년\1. 초장대 K-지하고속도로 인프라 안전 및 효율 향상 기술 개발\VISSIM_Workspace\vds\VDS_16_04_01_920560.csv
작업파일 :  E:\2026년\1. 초장대 K-지하고속도로 인프라 안전 및 효율 향상 기술 개발\VISSIM_Workspace\vds\VDS_16_04_01_502639.csv
작업파일 :  E:\2026년\1. 초장대 K-지하고속도로 인프라 안전 및 효율 향상 기술 개발\VISSIM_Workspace\vds\VDS_16_04_01_525254.csv
작업파일 :  E:\2026년\1. 초장대 K-지하고속도로 인프라 안전 및 효율 향상 기술 개발\VISSIM_Workspace\vds\VDS_16_04_01_425153.csv
작업파일 :  E: